# Recollect `download` -> `raw72`

In [10]:
import os
import time
import cosmic.utils as cu # type: ignore

def wait_for_enough_fits_files(target_dir: str, target_count: int = 1200) -> None:
    """ 等待目标目录下的.fits文件数量达到指定阈值（默认1200个）"""
    # 先检查目标目录是否存在，避免后续报错
    if not os.path.isdir(target_dir):
        raise ValueError(f"错误：目标目录 '{target_dir}' 不存在！")
    
    print(f"开始监控目录 '{target_dir}' 下的.fits文件，目标数量：{target_count}个")
    
    while True:
        # 统计目录下所有.fits文件（兼容大小写，且只统计文件，排除子目录）
        fits_count = 0
        for filename in os.listdir(target_dir):
            file_path = os.path.join(target_dir, filename)
            # 确保是文件 + 后缀是.fits（不区分大小写）
            if os.path.isfile(file_path) and filename.lower().endswith('.fits'):
                fits_count += 1
        
        # 检查是否达到目标数量
        if fits_count >= target_count:
            print(f"✅ 检测到{fits_count}个.fits文件，已达到目标数量，结束等待")
            break
        
        # 未达到目标数量，打印提示并等待10分钟（600秒）
        print(f"⚠️ 当前仅检测到{fits_count}个.fits文件（目标{target_count}个），将等待10分钟后重新检查...")
        time.sleep(600)  # 10分钟 = 600秒

In [11]:
target_directory = "/home/tiandc/Data/PanSTARRS/DR2All/download"
wait_for_enough_fits_files(target_directory)

input_dir = "/home/tiandc/Data/PanSTARRS/DR2All/download"
output_dir = "/home/tiandc/Data/PanSTARRS/DR2All/raw72"
cu.merge_fits_by_ra(
    input_dir=input_dir,
    output_dir=output_dir,
    output_filename="ps1dr2_{i:02d}.fits",
    temp_dir="./temp",
    num_processes=None,
    ra_column='raStack',
    fits_extension="*.fits",
    data_hdu_index=1,
    buffer_flush_size=2000000,
    bin_size=5.0,
    add_uid=True
)

开始监控目录 '/home/tiandc/Data/PanSTARRS/DR2All/download' 下的.fits文件，目标数量：1200个
✅ 检测到1200个.fits文件，已达到目标数量，结束等待
Step 1: Setting up directories...

Step 2: Preparing file list for 54 parallel processes...
Found 1200 source files.
Total data size: 232.60 GB

Step 3: Starting parallel processing...
Parallel processing finished in 32.48 minutes.

Step 4: Consolidating temporary files into final output files...


Consolidating bins: 100%|██████████| 72/72 [2:29:36<00:00, 124.67s/it]  



Step 5: Cleaning up temporary directory...

--- All Done! ---
Processing complete in 182.35 minutes.
Total rows processed: 680,467,576
Final files are located in: /home/tiandc/Data/PanSTARRS/DR2All/raw72


{'total_rows': 680467576,
 'duration_minutes': 182.3547573407491,
 'output_files': ['/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_00.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_01.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_02.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_03.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_04.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_05.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_06.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_07.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_08.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_09.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_10.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_11.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_12.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_13.fits',
  '/home/tiandc/Data/PanSTARRS/DR2All/raw72/ps1dr2_14.fits',
  '

In [12]:
# 收集 raw72 目录下所有文件的位置信息 [uid, raStack, decStack]
import os                                                                                                                                                                                            
import glob                                                                                                                                                                                          
import shutil                                                                                                                                                                                        
import pandas as pd                                                                                                                                                                                  
from tqdm import tqdm

raw72_dir = "/home/tiandc/Data/PanSTARRS/DR2All/raw72"
output_file = "/home/tiandc/Data/PanSTARRS/DR2All/ps1dr2_raw72_loc.fits"
temp_dir = "/home/tiandc/Data/PanSTARRS/DR2All/temp_loc"
save_interval = 10  # 每隔10个文件保存一次，释放内存

# 创建临时目录
os.makedirs(temp_dir, exist_ok=True)

# 获取所有 FITS 文件并按文件名排序
fits_files = sorted(glob.glob(os.path.join(raw72_dir, "*.fits")))
print(f"找到 {len(fits_files)} 个 FITS 文件")

# 要提取的列
cols_to_extract = ['uid', 'raStack', 'decStack']

# 读取并收集所有文件的指定列
subdfs = []
chunk_id = 0
temp_files = []

for i, fpath in enumerate(tqdm(fits_files, desc="读取文件"), start=1):
    df = cu.readfile(fpath)
    sub_df = df[cols_to_extract]
    subdfs.append(sub_df)
    
    # 每隔 save_interval 个文件保存一次并释放内存
    if i % save_interval == 0:
        merged = pd.concat(subdfs, ignore_index=True)
        temp_file = os.path.join(temp_dir, f"chunk_{chunk_id:03d}.fits")
        cu.savefile(merged, temp_file)
        temp_files.append(temp_file)
        print(f"\n[chunk {chunk_id}] 已保存 {len(merged):,} 行到临时文件")
        
        # 释放内存
        del subdfs, merged
        subdfs = []
        chunk_id += 1

# 处理剩余不足 save_interval 的文件
if subdfs:
    merged = pd.concat(subdfs, ignore_index=True)
    temp_file = os.path.join(temp_dir, f"chunk_{chunk_id:03d}.fits")
    cu.savefile(merged, temp_file)
    temp_files.append(temp_file)
    print(f"\n[chunk {chunk_id}] 已保存 {len(merged):,} 行到临时文件")
    del subdfs, merged

# 合并所有临时文件
print(f"\n正在合并 {len(temp_files)} 个临时文件...")
final_tables = []
for tf in tqdm(temp_files, desc="合并临时文件"):
    final_tables.append(cu.readfile(tf))

final_table = pd.concat(final_tables, ignore_index=True)
cu.savefile(final_table, output_file)
print(f"✅ 完成！已保存 {len(final_table):,} 行数据到 {output_file}")

# 清理临时文件
import shutil
shutil.rmtree(temp_dir)
print(f"已清理临时目录 {temp_dir}")

找到 72 个 FITS 文件


读取文件:  14%|█▍        | 10/72 [02:10<14:23, 13.93s/it]


[chunk 0] 已保存 57,859,521 行到临时文件


读取文件:  28%|██▊       | 20/72 [05:58<26:49, 30.94s/it]


[chunk 1] 已保存 81,551,443 行到临时文件


...


[chunk 6] 已保存 123,576,552 行到临时文件


读取文件: 100%|██████████| 72/72 [32:58<00:00, 27.48s/it]



[chunk 7] 已保存 12,951,217 行到临时文件

正在合并 8 个临时文件...


合并临时文件: 100%|██████████| 8/8 [00:56<00:00,  7.07s/it]


✅ 完成！已保存 680,467,576 行数据到 /home/tiandc/Data/PanSTARRS/DR2All/ps1dr2_raw72_loc.fits
已清理临时目录 /home/tiandc/Data/PanSTARRS/DR2All/temp_loc


# 去红化：`raw72` -> `raw72_dered`

已在`/home/tiandc/Data/LegacySurveys/DR9x10/code/dataProcess.ipynb`中操作

# 添加unWISE红外数据

- 先用stilts交叉匹配

In [2]:
import subprocess
import os

# === 配置参数 ===
# 建议使用绝对路径，防止报错
stilts_path = "/home/tiandc/download/stilts.jar" 
input1 = "/home/tiandc/Data/PanSTARRS/DR2All/ps1dr2_raw72_loc.fits"       
input2 = "/home/tiandc/Data/unWISE/unwisedr1_clean_all_loc.fits"
output_file = "/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/ps1dr2_raw72loc_xunWISEloc.fits"
heap_size = "128G"

custom_tmp_dir = "/home/tiandc/tmp" 

# 构建命令列表
cmd = [
    "java", f"-Xmx{heap_size}", f"-Djava.io.tmpdir={custom_tmp_dir}", "-jar", stilts_path, 
    "tmatch2",
    "runner=parallel16",          # 多线程
    f"in1={input1}",
    f"in2={input2}",
    f"out={output_file}",
    "matcher=sky",
    "params=1.0",                 # 匹配半径 1 arcsec
    "values1=raStack decStack",             
    "values2=ra dec",
    "join=1and2",                 # 1and2 = inner join
    "find=best",                  # 只要最佳匹配
    "progress=log"                # 显示进度
]

print("正在开始交叉匹配，请耐心等待...")
print("执行命令:", " ".join(cmd))

# 调用系统命令
# check=True 表示如果 STILTS 报错，Python 也会抛出异常停止运行
try:
    subprocess.run(cmd, check=True)
    print(f"\n匹配成功！结果已保存至: {output_file}")
except subprocess.CalledProcessError as e:
    print(f"\n匹配失败，错误代码: {e.returncode}")

正在开始交叉匹配，请耐心等待...
执行命令: java -Xmx128G -Djava.io.tmpdir=/home/tiandc/tmp -jar /home/tiandc/download/stilts.jar tmatch2 runner=parallel16 in1=/home/tiandc/Data/PanSTARRS/DR2All/ps1dr2_raw72_loc.fits in2=/home/tiandc/Data/unWISE/unwisedr1_clean_all_loc.fits out=/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/ps1dr2_raw72loc_xunWISEloc.fits matcher=sky params=1.0 values1=raStack decStack values2=ra dec join=1and2 find=best progress=log


Params: Max Error(Number)/arcsec=1.0
Tuning: HEALPix k(Integer)=14
Processing: Split, BasicParallel
Attempt to locate restricted common region
Assessing range of coordinates from table 1...................................
Coverage is: 0.7524007 of sky (HEALPix 1: ffff ffff 7777)
Assessing range of coordinates from table 2...................................
Coverage is: 0.99902344 of sky (HEALPix 1: ffff ffff ffff)
Potential match region: 0.75217694 of sky (HEALPix 1: ffff ffff 7777)
Counting rows in match region for table 1.....................................
680377098 rows in match region
Counting rows in match region for table 2.....................................
606448143 rows in match region
Binning rows for table 2......................................................
271394932/877843075 rows excluded (out of match region)
743799931 row refs for 877843075 rows in 642980315 bins
(average bin occupancy 1.1568005)
Scanning rows for table 1..........................................


匹配成功！结果已保存至: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/ps1dr2_raw72loc_xunWISEloc.fits


- 添加交叉到的unWISE数据，未交叉到的为NaN

In [1]:
# ============ 多线程版本 ============
import cosmic.utils as cu
import pandas as pd
import numpy as np
import gc
import os
from concurrent.futures import ThreadPoolExecutor
import threading

# ============ 星等转换函数 ============
def vega_to_ab(mag_vega, band):
    if band == 'w1':
        return mag_vega + 2.699
    elif band == 'w2':
        return mag_vega + 3.339
    else:
        raise ValueError('band must be w1 or w2')

def calc_ab_magerr(flux, flux_err):
    return 1.0857 * (flux_err / flux)

# Read the cross-match location file (shared, read-only)
# uid_1 = PS1DR2 uid, uid_2 = unWISE uid
path = "/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/ps1dr2_raw72loc_xunWISEloc.fits"
df_loc = cu.readfile(path)

# unWISE原始列（需要从unWISE文件中读取）
cols_unwise_raw = ['uid', 'mag_w1_vg', 'mag_w2_vg', 'flux_w1', 'flux_w2', 'dflux_w1', 'dflux_w2']

# 要添加到PS1DR2的新列（转换后）
cols_to_add = ['mag_w1', 'mag_w2', 'mag_w1_err', 'mag_w2_err']

# Create output directory
output_dir = '/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE/'
os.makedirs(output_dir, exist_ok=True)

# Thread-safe print lock
print_lock = threading.Lock()

def process_single_file(i):
    """Process a single file pair"""
    try:
        # Check if output file already exists, skip if so
        output_path = f'{output_dir}ps1dr2_{i:02d}_xunWISE.fits'
        if os.path.exists(output_path):
            with print_lock:
                print(f'[{i:02d}/71] Output file already exists, skipping.')
            return i, True, 'skipped'
        
        # Read current PS1DR2 raw72 file (ALL rows)
        ps_path = f'/home/tiandc/Data/PanSTARRS/DR2All/raw72_dered/ps1dr2_{i:02d}_dered.fits'
        df_ps = cu.readfile(ps_path)
        
        # Initialize new columns with NaN
        for col in cols_to_add:
            df_ps[col] = np.nan
        
        # Read current unWISE clean file
        unwise_path = f'/home/tiandc/Data/unWISE/clean/unwisedr1_clean_{i:02d}.fits'
        df_unwise = cu.readfile(unwise_path)
        
        # Select only needed columns and convert magnitudes
        df_unwise_subset = df_unwise[cols_unwise_raw].copy()
        del df_unwise
        
        # Convert Vega mag to AB mag and calculate errors
        df_unwise_subset['mag_w1'] = vega_to_ab(df_unwise_subset['mag_w1_vg'], 'w1')
        df_unwise_subset['mag_w2'] = vega_to_ab(df_unwise_subset['mag_w2_vg'], 'w2')
        df_unwise_subset['mag_w1_err'] = calc_ab_magerr(df_unwise_subset['flux_w1'], df_unwise_subset['dflux_w1'])
        df_unwise_subset['mag_w2_err'] = calc_ab_magerr(df_unwise_subset['flux_w2'], df_unwise_subset['dflux_w2'])
        
        # Keep only uid and converted columns
        df_unwise_subset = df_unwise_subset[['uid'] + cols_to_add]
        
        # Filter df_loc for current PS1DR2 file's uids
        # uid_1 is PS1DR2 uid, uid_2 is unWISE uid
        df_loc_subset = df_loc[df_loc['uid_1'].isin(df_ps['uid'])].copy()
        
        # Merge df_loc_subset with unWISE data to get mag columns
        df_match = df_loc_subset.merge(df_unwise_subset, left_on='uid_2', right_on='uid', how='left', suffixes=('', '_unwise'))
        del df_unwise_subset, df_loc_subset
        
        # Create a mapping from PS1DR2 uid to unWISE data
        df_match = df_match.set_index('uid_1')
        
        # Set PS1DR2 uid as index for efficient updating
        df_ps = df_ps.set_index('uid')
        
        # Update matched rows with unWISE data
        matched_uids = df_match.index.intersection(df_ps.index)
        for col in cols_to_add:
            df_ps.loc[matched_uids, col] = df_match.loc[matched_uids, col].values
        
        # Reset index to restore uid as a column
        df_ps = df_ps.reset_index()
        
        # Save to output directory
        cu.savefile(df_ps, output_path)
        
        # Report statistics
        n_total = len(df_ps)
        n_matched = len(matched_uids)
        
        with print_lock:
            print(f'[{i:02d}/71] Total: {n_total:,}, Matched: {n_matched:,} ({n_matched/n_total*100:.2f}%)')
        
        # Clean up memory
        del df_ps, df_match
        gc.collect()
        
        return i, True, None
    except Exception as e:
        with print_lock:
            print(f'[{i:02d}/71] Error: {e}')
        return i, False, str(e)

# Number of threads (adjust based on storage type)
# HDD: 2-4 threads, SSD: 6-10 threads
N_THREADS = 2

print(f'Processing 72 files with {N_THREADS} threads...')
print(f'Output: {output_dir}\n')

with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
    results = list(executor.map(process_single_file, range(72)))

# Summary
success = sum(1 for _, ok, _ in results if ok)
failed = [(i, err) for i, ok, err in results if not ok]

print(f'\n{"="*50}')
print(f'Completed: {success}/72')
if failed:
    print(f'Failed: {[i for i, _ in failed]}')

Processing 72 files with 2 threads...
Output: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE/

[00/71] Output file already exists, skipping.
[01/71] Output file already exists, skipping.
[02/71] Output file already exists, skipping.
[03/71] Output file already exists, skipping.
[04/71] Output file already exists, skipping.
[06/71] Output file already exists, skipping.
[05/71] Output file already exists, skipping.
[08/71] Output file already exists, skipping.
[07/71] Output file already exists, skipping.
[09/71] Output file already exists, skipping.
[10/71] Output file already exists, skipping.
[12/71] Output file already exists, skipping.
[11/71] Output file already exists, skipping.
[14/71] Output file already exists, skipping.
[13/71] Output file already exists, skipping.
[16/71] Output file already exists, skipping.
[15/71] Output file already exists, skipping.
[18/71] Output file already exists, skipping.
[17/71] Output file already exists, skipping.
[20/71] Output 

# 数据清理

In [9]:
import pandas as pd
import numpy as np 
import cosmic.utils as cu     
import os

def clean_panstarrs(df_raw):      
    """PanSTARRS 数据清理 - 保留高质量非恒星源"""   
                   
    df = df_raw.copy() 
    n_orig = len(df)
                              
    # 0. 先展平所有多维列     
    for col in df.columns:    
        if len(df[col].shape) > 1:                  
            df[col] = df[col][:, 0]                 
                   
    # 替换无效值为NaN
    invalid_vals = [-99.0, -99, -999.0, -9999.0, -999, -9999]
    df = df.replace(invalid_vals, np.nan)
    
    # 必须有效的核心列: grizy波段的PSF, Kron, Ap星等及其误差
    core_mag_cols = []
    core_err_cols = []
    for b in ['g', 'r', 'i', 'z', 'y']:
        core_mag_cols.extend([f'{b}PSFMag_dered', f'{b}KronMag_dered', f'{b}ApMag_dered'])
        core_err_cols.extend([f'{b}PSFMagErr', f'{b}KronMagErr', f'{b}ApMagErr'])
    
    existing_mag_cols = [c for c in core_mag_cols if c in df.columns]
    df = df.dropna(subset=existing_mag_cols)
    
    existing_err_cols = [c for c in core_err_cols if c in df.columns]
    df = df.dropna(subset=existing_err_cols)
                   
    # 1. nDetections >= 1 (至少有一次有效探测)
    df = df[np.array(df['nDetections']).flatten() >= 1]
                   
    # ============ objInfoFlag 过滤 ============    
    BAD_OBJ_MASK = 0x00000020 | 0x00000040 | 0x00080000 | 0x00100000      
    flags = np.nan_to_num(np.array(df['objInfoFlag']).flatten().astype(np.int64), nan=0)                   
    df = df[(flags & BAD_OBJ_MASK) == 0]
                   
    # ============ qualityFlag 过滤 ============    
    BAD_QUALITY_MASK = 0x00000040 | 0x00000080
    flags = np.nan_to_num(np.array(df['qualityFlag']).flatten().astype(np.int64), nan=0)
    df = df[(flags & BAD_QUALITY_MASK) == 0]
                   
    # ============ infoFlag 过滤 (每个波段) ============                  
    STAR_MASK = 0x400000
    CONTAMINATION_MASK = 0x8 | 0x400 | 0x800 | 0x1000 | 0x2000 | 0x10000
    EXCLUDE_MASK = STAR_MASK | CONTAMINATION_MASK   
                   
    mask = np.ones(len(df), dtype=bool)             
    for b in ['g', 'r', 'i', 'z', 'y']:             
        col = f'{b}infoFlag'  
        if col in df.columns: 
            flags = np.nan_to_num(np.array(df[col]).flatten().astype(np.int64), nan=0)                     
            mask &= (flags & EXCLUDE_MASK) == 0
    df = df[mask]
    
    # ============ 恒星排除 (基于PSF-Kron星等差) ============
    if 'iPSFMag_dered' in df.columns and 'iKronMag_dered' in df.columns:
        i_psf = df['iPSFMag_dered'].values
        i_kron = df['iKronMag_dered'].values
        psf_kron_diff = i_psf - i_kron
        is_bright = i_psf < 21
        is_star = psf_kron_diff <= 0.05
        df = df[~is_bright | ~is_star]
                   
    # 重命名列     
    df = df.rename(columns={'raStack': 'ra', 'decStack': 'dec'})                  
                   
    # 指定要保留的列          
    cols_to_keep = [          
        'objID', 'uid', 'ra', 'dec',             
        'gKronMag_dered', 'rKronMag_dered', 'iKronMag_dered', 'zKronMag_dered', 'yKronMag_dered',          
        'gPSFMag_dered', 'rPSFMag_dered', 'iPSFMag_dered', 'zPSFMag_dered', 'yPSFMag_dered',               
        'gApMag_dered', 'rApMag_dered', 'iApMag_dered', 'zApMag_dered', 'yApMag_dered',                    
        'gKronMagErr', 'rKronMagErr', 'iKronMagErr', 'zKronMagErr', 'yKronMagErr',   
        'gPSFMagErr', 'rPSFMagErr', 'iPSFMagErr', 'zPSFMagErr', 'yPSFMagErr',        
        'gApMagErr', 'rApMagErr', 'iApMagErr', 'zApMagErr', 'yApMagErr',  
        'mag_w1', 'mag_w2', 'mag_w1_err', 'mag_w2_err',
    ]              
    cols_to_keep = [c for c in cols_to_keep if c in df.columns]           
    df_clean = df[cols_to_keep].reset_index(drop=True)                    
    
    print(f'Clean: {n_orig:,} -> {len(df_clean):,} ({len(df_clean)/n_orig*100:.2f}%)')        
                   
    return df_clean

In [10]:
import os 
output_dir = '/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/'
os.makedirs(output_dir, exist_ok=True)

for fid in range(72):
    path = f'/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE/ps1dr2_{fid:02d}_xunWISE.fits'
    output_path = os.path.join(output_dir, f'ps1dr2_{fid:02d}_xunWISE_clean.fits')
    if os.path.exists(output_path):
        print(f'[{fid}/71] exists: pass')
        continue
    
    df = cu.readfile(path)
    df_clean = clean_panstarrs(df)
    
    cu.savefile(df_clean, output_path)
    print(f'[{fid}/71] Saved cleaned file: {output_path}')

Clean: 6,210,999 -> 1,978,816 (31.86%)
[0/71] Saved cleaned file: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_00_xunWISE_clean.fits
Clean: 6,019,310 -> 1,996,885 (33.17%)
[1/71] Saved cleaned file: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_01_xunWISE_clean.fits
Clean: 5,998,060 -> 1,993,398 (33.23%)
[2/71] Saved cleaned file: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_02_xunWISE_clean.fits
Clean: 5,787,542 -> 1,911,856 (33.03%)
[3/71] Saved cleaned file: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_03_xunWISE_clean.fits
Clean: 5,778,500 -> 1,830,582 (31.68%)
[4/71] Saved cleaned file: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_04_xunWISE_clean.fits
Clean: 5,758,863 -> 1,817,812 (31.57%)
[5/71] Saved cleaned file: /home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_05_xunWISE_clean.fits
Clean: 5,753,121

# 测试

In [7]:
import cosmic.utils as cu
from astropy.table import Table
path = "/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE_clean/ps1dr2_00_xunWISE_clean.fits"
tab = Table.read(path)

In [8]:
tab

objID,uid_ps,ra,dec,gKronMag_dered,rKronMag_dered,iKronMag_dered,zKronMag_dered,yKronMag_dered,gPSFMag_dered,rPSFMag_dered,iPSFMag_dered,zPSFMag_dered,yPSFMag_dered,gApMag_dered,rApMag_dered,iApMag_dered,zApMag_dered,yApMag_dered,gKronMagErr,rKronMagErr,iKronMagErr,zKronMagErr,yKronMagErr,gPSFMagErr,rPSFMagErr,iPSFMagErr,zPSFMagErr,yPSFMagErr,gApMagErr,rApMagErr,iApMagErr,zApMagErr,yApMagErr,mag_w1,mag_w2,mag_w1_err,mag_w2_err
int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
72000011574396493,0,1.15737944,-29.99482873,21.429575871676207,20.369610326364636,19.63297407515347,19.456502065062523,19.406618556939065,21.985976170748472,20.778511540964246,20.158273147419095,19.890002354979515,19.713320217095315,21.51047606393695,20.337111013010144,19.676673339679837,19.445401296019554,19.245918712578714,0.12112999707460403,0.035078998655080795,0.021511999890208244,0.03053100034594536,0.07660499960184097,0.10444299876689911,0.028224000707268715,0.01654299907386303,0.024109000340104103,0.05924500152468681,0.008333000354468822,0.00407300004735589,0.002303000073879957,0.0034030000679194927,0.0029980000108480453,18.488928436279295,18.840086235046385,0.01808784411661327,0.051202238480374224
72000020983856336,1,2.09832595,-29.99506228,22.024601083248854,21.403092063963413,20.272807840257883,20.18758046068251,19.834099761210382,22.68210230395198,21.75979296118021,20.755208734422922,20.51577984727919,20.23889826890081,22.01300249621272,21.290293373167515,20.249507669359446,20.101379746571183,19.67919825669378,0.14395099878311157,0.0810369998216629,0.028442999348044395,0.06105799973011017,0.09401600062847137,0.13658000528812408,0.06092499941587448,0.02397800050675869,0.04341999813914299,0.07762899994850159,0.013371000066399574,0.007911000400781631,0.003487000009045005,0.0057830000296235085,0.004279999993741512,18.975943206787107,19.05509878540039,0.026799393335916105,0.060408005136251455
72000022888767732,2,2.28887622,-29.99379445,18.97952199727297,18.527983598411083,18.184990648180246,18.091528926044703,18.148283371701837,20.13592267781496,19.71558278053999,19.326290849596262,19.452029261738062,19.264383682981133,19.257323510944843,18.905283860862255,18.52929187193513,18.568030390888453,18.455084213986993,0.012707999907433987,0.011729000136256218,0.01058799959719181,0.016961000859737396,0.028543999418616295,0.013675999827682972,0.011152000166475773,0.009573999792337418,0.016959000378847122,0.030872000381350517,0.0037930000107735395,0.002297000028192997,0.0014690000098198652,0.0028719999827444553,0.002449000021442771,18.60362779998779,18.719431175231932,0.02029578898306936,0.04577111470140517
72000036800266163,3,3.68011841,-29.9950355,20.54929630830884,19.922036197036505,19.584959719330072,19.51216840185225,19.40856882929802,21.212197232991457,20.706635501235723,20.321959231048822,20.42696904577315,20.292369335889816,20.53859608247876,19.984034564346075,19.661858294159174,19.48626851476729,19.368169277906418,0.03093699924647808,0.026094000786542892,0.019186999648809433,0.04706300050020218,0.08639399707317352,0.03245300054550171,0.02568499930202961,0.017952999100089073,0.05171100050210953,0.09292799979448318,0.005956000182777643,0.004077000077813864,0.002515000058338046,0.0047690002247691154,0.004011999815702438,19.649443267822264,19.876994384765624,0.04825244470126928,0.128849565897882
72000040173601041,4,4.01733588,-29.99938082,20.175549797713757,19.60017691925168,19.309134462848306,19.211199523881078,19.11549098789692,20.764149956405163,20.285576928406954,19.959235170856118,19.909397842362523,19.811192587018013,20.174550347030163,19.694975960999727,19.325335482135415,19.23269915767014,19.107392385601997,0.02908400073647499,0.01827099919319153,0.01669199950993061,0.033608999103307724,0.06211699917

In [19]:
df.groupby('priority').size()

priority
True    11328474
dtype: int64

In [23]:
import numpy as np                                                                                                                                                                                                          
import pandas as pd                                                                                                                                                                                                         
                                                                                                                                                                                                                            
# 模拟 predict_lsdr9x10 的逻辑                                                                                                                                                                                              
n_samples = 10                                                                                                                                                                                                              
priority_array = np.zeros(n_samples, dtype=np.int8)                                                                                                                                                                         
survey_array = np.zeros(n_samples, dtype=np.int8)                                                                                                                                                                           
                                                                                                                                                                                                                            
# 模拟掩码                                                                                                                                                                                                                  
has_native_i = np.array([True, True, False, False, True, False, False, True, False, False])                                                                                                                                 
has_ps1_i = np.array([False, True, True, False, False, True, False, False, True, False])                                                                                                                                    
                                                                                                                                                                                                                            
mask_p1 = has_native_i                                                                                                                                                                                                      
mask_p2 = ~has_native_i & has_ps1_i                                                                                                                                                                                         
mask_p3 = ~has_native_i & ~has_ps1_i                                                                                                                                                                                        
                                                                                                                                                                                                                            
model_configs = [                                                                                                                                                                                                           
    ('p1', mask_p1, 1),                                                                                                                                                                                                     
    ('p2', mask_p2, 2),                                                                                                                                                                                                     
    ('p3', mask_p3, 3),                                                                                                                                                                                                     
]                                                                                                                                                                                                                           
                                                                                                                                                                                                                            
print("Before loop:")                                                                                                                                                                                                       
print(f"  priority_array dtype: {priority_array.dtype}")                                                                                                                                                                    
print(f"  priority_array: {priority_array}")                                                                                                                                                                                
                                                                                                                                                                                                                            
for model_name, mask, priority in model_configs:                                                                                                                                                                            
    indices = np.where(mask)[0]                                                                                                                                                                                             
    print(f"\n{model_name}: indices={indices}, priority={priority}, type(priority)={type(priority)}")                                                                                                                       
    priority_array[indices] = priority                                                                                                                                                                                      
    print(f"  After assignment: {priority_array}")                                                                                                                                                                          
                                                                                                                                                                                                                            
print(f"\nFinal priority_array: {priority_array}")                                                                                                                                                                          
print(f"Final dtype: {priority_array.dtype}")                                                                                                                                                                               
                                                                                                                                                                                                                            
# 构建 DataFrame                                                                                                                                                                                                            
result = pd.DataFrame({                                                                                                                                                                                                     
    'priority': priority_array,                                                                                                                                                                                             
    'survey': survey_array,                                                                                                                                                                                                 
})                                                                                                                                                                                                                          
print(f"\nDataFrame:\n{result}")                                                                                                                                                                                            
print(f"\nDataFrame dtypes:\n{result.dtypes}")

Before loop:
  priority_array dtype: int8
  priority_array: [0 0 0 0 0 0 0 0 0 0]

p1: indices=[0 1 4 7], priority=1, type(priority)=<class 'int'>
  After assignment: [1 1 0 0 1 0 0 1 0 0]

p2: indices=[2 5 8], priority=2, type(priority)=<class 'int'>
  After assignment: [1 1 2 0 1 2 0 1 2 0]

p3: indices=[3 6 9], priority=3, type(priority)=<class 'int'>
  After assignment: [1 1 2 3 1 2 3 1 2 3]

Final priority_array: [1 1 2 3 1 2 3 1 2 3]
Final dtype: int8

DataFrame:
   priority  survey
0         1       0
1         1       0
2         2       0
3         3       0
4         1       0
5         2       0
6         3       0
7         1       0
8         2       0
9         3       0

DataFrame dtypes:
priority    int8
survey      int8
dtype: object


In [24]:
result

,priority,survey
0,1,0
1,1,0
2,2,0
3,3,0
4,1,0
5,2,0
6,3,0
7,1,0
8,2,0
9,3,0


In [5]:
import pandas as pd
from astropy.table import Table
import os

def read_fits(filepath: str) -> pd.DataFrame:
    """读取 FITS 文件为 DataFrame"""
    table = Table.read(filepath, format='fits')
    return table.to_pandas()

def read_table(filepath: str) -> pd.DataFrame:
    """根据扩展名读取表。"""
    ext = os.path.splitext(filepath)[1].lower()
    if ext in ['.fits', '.fit', '.fz']:
        return read_fits(filepath)
    raise ValueError(f"Unsupported input format: {filepath}")

path = '/home/tiandc/Data/PanSTARRS/DR2All/xunWISE/raw72_dered_xunWISE/ps1dr2_00_xunWISE.fits'
df = read_table(path)

In [6]:
df.columns

Index(['uid', 'objID', 'raStack', 'decStack', 'l', 'b', 'nDetections',
       'gPSFMag_dered', 'gPSFMagErr', 'gApMag_dered', 'gApMagErr',
       'gKronMag_dered', 'gKronMagErr', 'rPSFMag_dered', 'rPSFMagErr',
       'rApMag_dered', 'rApMagErr', 'rKronMag_dered', 'rKronMagErr',
       'iPSFMag_dered', 'iPSFMagErr', 'iApMag_dered', 'iApMagErr',
       'iKronMag_dered', 'iKronMagErr', 'zPSFMag_dered', 'zPSFMagErr',
       'zApMag_dered', 'zApMagErr', 'zKronMag_dered', 'zKronMagErr',
       'yPSFMag_dered', 'yPSFMagErr', 'yApMag_dered', 'yApMagErr',
       'yKronMag_dered', 'yKronMagErr', 'ginfoFlag', 'ginfoFlag2',
       'ginfoFlag3', 'rinfoFlag', 'rinfoFlag2', 'rinfoFlag3', 'iinfoFlag',
       'iinfoFlag2', 'iinfoFlag3', 'zinfoFlag', 'zinfoFlag2', 'zinfoFlag3',
       'yinfoFlag', 'yinfoFlag2', 'yinfoFlag3', 'qualityFlag', 'objInfoFlag',
       'mag_w1', 'mag_w2', 'mag_w1_err', 'mag_w2_err'],
      dtype='object')

In [4]:
print(df.uid.duplicated().sum())
print(df.uid.isna().sum())

0
0
